# Lesson 3.4: Dynamic Agents

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

# 1. Dynamic agents

In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

standard_model = init_chat_model(
    model="openai/gpt-oss-120b", 
    model_provider="groq"
)

# 2. Long Conversation Model (Massive Context Gemini)
large_model = init_chat_model(
    model="models/gemini-3.5-flash-lite", 
    model_provider="google_genai"
)


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model=standard_model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

Hey! I gave our office plant a good drink this morning—about a cup of water, making sure the soil is evenly moist but not soggy. It looks happy and perkier already! 🌿 Let me know if there’s anything else I can take care of today.


In [7]:
print(response["messages"][-1].response_metadata["model_name"])

openai/gpt-oss-120b


In [8]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content[0]['text'])

Hmm, I'm actually not totally sure! Probably not for a while, since it still has some room to grow. Do you want me to check the roots to see if it's getting root-bound, or look it up online?


In [9]:
print(response["messages"][-1].response_metadata["model_name"])

gemini-3.5-flash-lite


# 2. Dynamic prompts

In [10]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    if request.runtime.context is not None:
        user_language = request.runtime.context.user_language
        base_prompt = "You are a helpful assistant."
 
        if user_language != "English":
            return f"{base_prompt} only respond in {user_language}."
        elif user_language == "English":
            return base_prompt
    else:
        return "This should never happen, but if it does, just respond in English."

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model=standard_model,
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [12]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

Dia dhuit! Tá mé go maith, go raibh maith agat. Conas atá tú?


In [13]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

¡Hola! Estoy bien, gracias. ¿Y tú?


In [14]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

Bonjour ! Je vais très bien, merci. Et vous, comment allez‑vous ?


In [15]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="English")
)

print(response["messages"][-1].content)

Hello! I'm doing great, thank you for asking. How can I assist you today?


# 3. Dynamic tools

In [16]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

C:\Users\reich\AppData\Local\Temp\ipykernel_19132\1699794177.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [17]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [18]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [19]:
from langchain.agents import create_agent

agent = create_agent(
    model=standard_model,
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [20]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

The current MusicBrainz database contains **about 2,969,628 artists** (including persons, groups, orchestras, choirs, characters, etc.)【2†L1-L8】.
